In [1]:
pip install spotipy pandas requests python-dotenv matplotlib

Note: you may need to restart the kernel to use updated packages.


Este código solo da los top artistas, sin género (no requiere de LastFM):

In [ ]:
# import pandas as pd
   
# RANGOS = {
#     "short_term": "ultimas_4_semanas",
#     "medium_term": "ultimos_6_meses",
#     "long_term": "varios_anios",
# }
 
# filas = []
# for rango, etiqueta in RANGOS.items():
#     resultado = sp.current_user_top_artists(limit=50, time_range=rango)
#     for posicion, artista in enumerate(resultado["items"], start=1):
#         filas.append(
#             {
#                 "rango": etiqueta,
#                 "posicion": posicion,
#                 "artista": artista["name"],
#                 "generos": "; ".join(artista.get("genres", [])),
#                 "spotify_url": artista["external_urls"]["spotify"],
#             }
#         )
 
# df = pd.DataFrame(filas)
# df.to_csv("top_artistas.csv", index=False, encoding="utf-8-sig")
 
# # Vista rápida: top 10 de cada rango
# for etiqueta in RANGOS.values():
#     print(f"\n=== {etiqueta} ===")
#     top = df[df["rango"] == etiqueta].head(10)
#     for _, fila in top.iterrows():
#         print(f"{fila['posicion']:>2}. {fila['artista']}")
 
# print("\nGuardado en top_artistas.csv")


=== ultimas_4_semanas ===
 1. Toby Fox
 2. Linkin Park
 3. SEGA SOUND TEAM
 4. Bring Me The Horizon
 5. アトラスサウンドチーム
 6. Melendi
 7. My Chemical Romance
 8. Fito y Fitipaldis
 9. Architects
10. Franz Ferdinand

=== ultimos_6_meses ===
 1. Toby Fox
 2. Linkin Park
 3. Fito y Fitipaldis
 4. SEGA SOUND TEAM
 5. Bring Me The Horizon
 6. アトラスサウンドチーム
 7. My Chemical Romance
 8. Melendi
 9. The Killers
10. Franz Ferdinand

=== varios_anios ===
 1. Linkin Park
 2. アトラスサウンドチーム
 3. SEGA SOUND TEAM
 4. Toby Fox
 5. Bring Me The Horizon
 6. My Chemical Romance
 7. 平田志穂子
 8. Lyn
 9. Franz Ferdinand
10. Fito y Fitipaldis

Guardado en top_artistas.csv


Este da el top artistas con 4 géneros (se pueden pedir más géneros, requiere de LastFM):

In [8]:
"""
Guarda tus artistas más escuchados de Spotify, con géneros de Last.fm,
en un único CSV con las columnas:
    rango,posicion,artista,generos,spotify_url

Requisitos:
    pip install spotipy pandas requests python-dotenv

Fichero .env en la misma carpeta:
    SPOTIPY_CLIENT_ID=tu_client_id
    SPOTIPY_CLIENT_SECRET=tu_client_secret
    SPOTIPY_REDIRECT_URI=http://127.0.0.1:8888/callback
    LASTFM_API_KEY=tu_api_key
"""

import os
import time

import pandas as pd
import requests
import spotipy
from dotenv import load_dotenv
from spotipy.oauth2 import SpotifyOAuth

load_dotenv()

LASTFM_KEY = os.getenv("LASTFM_API_KEY")
if not LASTFM_KEY:
    raise SystemExit("Falta LASTFM_API_KEY en el fichero .env")

RANGOS = {
    "short_term": "ultimas_4_semanas",
    "medium_term": "ultimos_6_meses",
    "long_term": "varios_anios",
}

MIN_PESO = 20     # peso mínimo (0-100) de una etiqueta de Last.fm
MAX_GENEROS = 4   # géneros guardados por artista

# Etiquetas de Last.fm que no son géneros musicales
IGNORAR = {
    "seen live", "female vocalists", "male vocalists", "american", "british",
    "spanish", "french", "german", "swedish", "canadian", "australian",
    "singer-songwriter", "favorites", "favourite", "awesome", "legend",
}


def generos_de(artista: str) -> str:
    params = {
        "method": "artist.gettoptags",
        "artist": artista,
        "api_key": LASTFM_KEY,
        "format": "json",
        "autocorrect": 1,
    }
    try:
        r = requests.get("https://ws.audioscrobbler.com/2.0/", params=params, timeout=10)
        r.raise_for_status()
        etiquetas = r.json().get("toptags", {}).get("tag", [])
    except (requests.RequestException, ValueError):
        return ""

    generos = []
    for e in etiquetas:
        nombre = e["name"].strip().lower()
        if int(e.get("count", 0)) >= MIN_PESO and nombre not in IGNORAR:
            generos.append(nombre)
        if len(generos) == MAX_GENEROS:
            break
    return "; ".join(generos)


sp = spotipy.Spotify(auth_manager=SpotifyOAuth(scope="user-top-read"))

filas = []
for rango, etiqueta in RANGOS.items():
    resultado = sp.current_user_top_artists(limit=50, time_range=rango)
    for posicion, artista in enumerate(resultado["items"], start=1):
        filas.append(
            {
                "rango": etiqueta,
                "posicion": posicion,
                "artista": artista["name"],
                "spotify_url": artista["external_urls"]["spotify"],
            }
        )

df = pd.DataFrame(filas)

# Una sola consulta a Last.fm por artista, aunque salga en varios rangos
cache = {}
for nombre in df["artista"].unique():
    cache[nombre] = generos_de(nombre)
    print(f"{nombre}: {cache[nombre] or '(sin géneros)'}")
    time.sleep(0.25)  # respeta el límite de peticiones

df["generos"] = df["artista"].map(cache)

df = df[["rango", "posicion", "artista", "generos", "spotify_url"]]
df.to_csv("top_artistas.csv", index=False, encoding="utf-8-sig")
print("\nGuardado en top_artistas.csv")

Toby Fox: soundtrack; chiptune; video game music; electronic
Linkin Park: nu metal; rock; alternative rock; alternative
SEGA SOUND TEAM: video game music; soundtrack
Bring Me The Horizon: deathcore; metalcore; screamo; hardcore
アトラスサウンドチーム: acid jazz; japanese; video game music
Melendi: pop; flamenco; new flamenco
My Chemical Romance: rock; alternative; punk rock; emo
Fito y Fitipaldis: rock; blues; spanish rock; blues rock
Architects: metalcore; mathcore; hardcore
Franz Ferdinand: indie rock; indie; rock; alternative
平田志穂子: soundtrack; japanese; j-pop
Azumi Takahashi: japanese; soul
Pentakill: power metal; heavy metal; metal
Lyn: korean; k-pop; ballad; pop
Lotus Juice: hip-hop; japanese; electronic; rap
League of Legends: soundtrack
El Sueño de Morfeo: pop; rock
ABBA: pop; disco; 70s
Maroon 5: pop rock; rock; pop; alternative
Gorillaz: electronic; alternative; hip-hop; rock
Korn: nu metal; metal; alternative; rock
Lorien Testard: soundtrack
Arctic Monkeys: indie rock; indie; alternati